# 逐次予測

前処理済みのデータを用いてモデルを構築し、評価する

- 目的変数: `price_actual`
- モデル: LightGBM, RandomForest, SVR, NeuralNetwork のいずれか
- 評価指標: RMSE
- ハイパーパラメータチューニング: ベイズ最適化

## 1. ライブラリのインポートとデータ読み込み

In [1]:
# 自動ローディング
%load_ext autoreload
%autoreload 2

In [2]:
# アクティベート
!source ../../.venv/bin/activate

In [3]:
# LightGBM特有のエラー対策
#!brew install libomp
#!pip uninstall lightgbm
#!pip install lightgbm

In [4]:
import pandas as pd
import numpy as np
from pathlib import Path

# データ読み込み用の関数をインポート
import sys, os
sys.path.append(os.pardir)  # 親ディレクトリのファイルをインポートするための設定
from src.data_loader import check_missing_values
from src.modeling import train_and_predict

# データディレクトリ
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / 'data'
SUBMISSION = DATA_DIR / 'submission'

os.makedirs(SUBMISSION, exist_ok=True)

print(DATA_DIR)
# 前処理済みデータの読み込み
train = pd.read_csv(DATA_DIR / 'train_processed.csv')
test = pd.read_csv(DATA_DIR / 'test_processed.csv')

print('train shape:', train.shape)
print('test shape:', test.shape)

/Users/m0122wt/Desktop/02.プライベート/01.ノウハウ/07.データ分析/notebook/signate_smbc_202506/data
train shape: (26280, 94)
test shape: (8760, 93)


## 2. 特徴量・目的変数の設定

In [5]:
# 目的変数
target_col = 'price_actual'

# 説明変数（目的変数とtime列以外）
drop_cols = ['time', target_col] if target_col in train.columns else ['time']
feature_cols = [col for col in train.columns if col not in drop_cols]

X = train[feature_cols]
y = train[target_col] if target_col in train.columns else train.iloc[:, -1]  # 念のため

print('Features:', feature_cols)
print('Target:', target_col)
print('X shape:', X.shape)
print('y shape:', y.shape)

Features: ['generation_biomass', 'generation_fossil_brown_coal/lignite', 'generation_fossil_gas', 'generation_fossil_hard_coal', 'generation_fossil_oil', 'generation_hydro_pumped_storage_consumption', 'generation_hydro_run_of_river_and_poundage', 'generation_hydro_water_reservoir', 'generation_nuclear', 'generation_other', 'generation_other_renewable', 'generation_solar', 'generation_waste', 'generation_wind_onshore', 'total_load_actual', 'valencia_pressure', 'valencia_humidity', 'valencia_wind_speed', 'valencia_wind_deg', 'valencia_rain_1h', 'valencia_rain_3h', 'valencia_snow_3h', 'valencia_clouds_all', 'madrid_pressure', 'madrid_humidity', 'madrid_wind_speed', 'madrid_wind_deg', 'madrid_rain_1h', 'madrid_rain_3h', 'madrid_snow_3h', 'madrid_clouds_all', 'bilbao_pressure', 'bilbao_humidity', 'bilbao_wind_speed', 'bilbao_wind_deg', 'bilbao_rain_1h', 'bilbao_rain_3h', 'bilbao_snow_3h', 'bilbao_clouds_all', 'barcelona_pressure', 'barcelona_humidity', 'barcelona_wind_speed', 'barcelona_win

In [6]:
# 学習とテストデータでカラムの構成に違いがないか確認
diff_features = set(train.columns) - set(test.columns)
print("差分があるカラム:")
print(sorted(list(diff_features)))

# 欠損値の確認
check_missing_values(train)
check_missing_values(test)

差分があるカラム:
['price_actual']
欠損値があるカラム、欠損値の数、全レコードに対する割合:


,missing_count,missing_ratio


欠損値があるカラム、欠損値の数、全レコードに対する割合:


,missing_count,missing_ratio


## 3. 学習・検証

In [ ]:
%%time
filename = 'submission_nn'

# 通常の予測
model, predictions = train_and_predict(
    model_type='neural_network',
    train_df=train,
    test_df=test,
    target_col='price_actual',
    optimize=True,
    sequential=False,  # 通常予測
    output_path=SUBMISSION,
    filename=filename
)

### 履歴
___
LIGHTGBM：ベイズ最適化の結果
Best params: {'num_leaves': 20, 'learning_rate': 0.07056499143559153, 'feature_fraction': 0.930666633751073, 'bagging_fraction': 0.8764523692423547, 'bagging_freq': 9, 'min_child_samples': 35, 'objective': 'regression', 'metric': 'rmse', 'boosting_type': 'gbdt', 'verbose': -1, 'random_state': 42}  
Best CV RMSE: 2.517122779389563  
Selected features: 73/92  
___
NEURAL_NETWORK：ベイズ最適化の結果
Best params: {'layer1_units': 48, 'layer2_units': 121, 'layer3_units': 8, 'dropout_rate': 0.16207371516723865, 'learning_rate': 0.005173979484789956, 'batch_size': 16}  
Best CV RMSE: 3.7179227519163134  
___


In [ ]:
predictions

In [ ]:
%%time

filename_sequential = 'submission_nn_sequential'

# 逐次予測（ラグ特徴量を考慮）
model_sequential, predictions_sequential = train_and_predict(
    model_type='neural_network',
    train_df=train,
    test_df=test,
    target_col='price_actual',
    optimize=True,
    sequential=True,  # 逐次予測
    output_path=SUBMISSION,
    filename=filename_sequential
)

[I 2025-06-29 16:16:17,234] A new study created in memory with name: no-name-50bb1049-8f21-4195-a625-62b95eff998c


137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 666us/step
137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 867us/step
137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 636us/step
137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 746us/step
137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 733us/step


[I 2025-06-29 16:18:38,151] Trial 0 finished with value: 5.074263028514065 and parameters: {'layer1_units': 231, 'layer2_units': 119, 'layer3_units': 41, 'dropout_rate': 0.45816053802580203, 'learning_rate': 0.0017795345905176271, 'batch_size': 16}. Best is trial 0 with value: 5.074263028514065.


137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 572us/step
137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 717us/step
137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 855us/step
137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 610us/step
137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 672us/step


[I 2025-06-29 16:21:10,988] Trial 1 finished with value: 4.793218494337585 and parameters: {'layer1_units': 72, 'layer2_units': 85, 'layer3_units': 8, 'dropout_rate': 0.3424013268551534, 'learning_rate': 0.007644347243250683, 'batch_size': 16}. Best is trial 1 with value: 4.793218494337585.


...

137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 465us/step
137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 501us/step
137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 497us/step
137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 478us/step
137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 523us/step


[I 2025-06-29 17:34:48,825] Trial 46 finished with value: 4.973814029113038 and parameters: {'layer1_units': 151, 'layer2_units': 32, 'layer3_units': 31, 'dropout_rate': 0.1862350509185439, 'learning_rate': 0.008582523127940705, 'batch_size': 64}. Best is trial 42 with value: 3.803042084025541.


137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 488us/step
137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 498us/step
137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 516us/step
137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 479us/step
137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 536us/step


[I 2025-06-29 17:36:55,612] Trial 47 finished with value: 4.561239355043058 and parameters: {'layer1_units': 199, 'layer2_units': 69, 'layer3_units': 22, 'dropout_rate': 0.2817386115484715, 'learning_rate': 0.003385431189179845, 'batch_size': 16}. Best is trial 42 with value: 3.803042084025541.


137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 501us/step
137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 441us/step


### 履歴
___
LIGHTGBM：ベイズ最適化の結果
Best params: {'num_leaves': 21, 'learning_rate': 0.0926973092334304, 'feature_fraction': 0.9533804854356259, 'bagging_fraction': 0.9723501183105292, 'bagging_freq': 10, 'min_child_samples': 43, 'objective': 'regression', 'metric': 'rmse', 'boosting_type': 'gbdt', 'verbose': -1, 'random_state': 42}  
Best CV RMSE: 2.51734282613965  
Selected features: 73/92
___

In [ ]:
predictions_sequential